<a href="https://colab.research.google.com/github/PanditPranav/WildAlertModels_Circumstances/blob/main/notebooks/04_04102024_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, PretrainedConfig

import pandas as pd
import torch
import torch.nn.functional as F
from datetime import datetime
# extract date object
today = datetime.now().date().strftime("%d-%m-%y")
# format date
print('Date String', today)

Date String 09-04-26


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from pathlib import Path
model_dir = Path("/content/drive/MyDrive/WildAlertCOA/WildAlertCircumstancesExp")
data_dir = Path("/content/drive/MyDrive/WildAlertCOA/data/processed")
raw_data_dir = Path("/content/drive/MyDrive/WildAlertCOA/data/raw")
interim_data_dir = Path("/content/drive/MyDrive/WildAlertCOA/data/interim")

data = pd.read_parquet (raw_data_dir/"wildalert_circumstances_api.parquet")

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_dir/"checkpoint-38600/")
model = AutoModelForSequenceClassification.from_pretrained(model_dir/"checkpoint-38600/")
config = PretrainedConfig.from_pretrained(model_dir/"checkpoint-38600/")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

You are using a model of type bert to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


In [6]:
sample = "White-fronted Amazon. found with illegal loggers. unknown, Illegal logging in Crooked Tree. Victoria confiscation while Nikki & Celeshia in OW doing checks"

In [7]:
device = torch.device("cuda")
model = model.to(device)

In [8]:
print(model.name_or_path)

/content/drive/MyDrive/WildAlertCOA/WildAlertCircumstancesExp/checkpoint-38600


In [9]:
def _pre(text, device):
    tokens = tokenizer(text, return_tensors="pt")
    tokens = {k: v.to(device) for k,v in tokens.items()}
    return tokens

def _inf(model, tokens):
    out = model(**tokens)
    return out

def _post(out):
    probs = F.sigmoid(out.logits.squeeze().detach().cpu())
    preds = (probs > 0.5).int()
    labels = [config.id2label[idx] for idx, label in enumerate(preds) if label == 1.0]
    return labels, probs

def infer(model, text):
    tokens = _pre(text, model.device)
    out = _inf(model, tokens)
    pred, probs = _post(out)
    return pred, [f'{x:f}' for x in probs], model.name_or_path


In [10]:
import pandas as pd
pd.options.display.float_format = '{:.2f}'.format

In [11]:
infer(model, sample)

(['confiscation'],
 ['0.000010',
  '0.000009',
  '0.000000',
  '0.000002',
  '0.000000',
  '0.000000',
  '0.000025',
  '0.000007',
  '0.999553',
  '0.000000',
  '0.000028',
  '0.000002',
  '0.000001',
  '0.000018',
  '0.000002',
  '0.000000',
  '0.000035',
  '0.000018',
  '0.000001',
  '0.000003',
  '0.000000',
  '0.000001',
  '0.000003',
  '0.000000',
  '0.000000',
  '0.000005',
  '0.000000',
  '0.000054',
  '0.000000',
  '0.000000',
  '0.000000',
  '0.000023',
  '0.000003',
  '0.000002',
  '0.000029',
  '0.000009',
  '0.000006',
  '0.000000',
  '0.000020',
  '0.000007',
  '0.000000',
  '0.000015',
  '0.000057',
  '0.000001',
  '0.000054',
  '0.000001',
  '0.000002',
  '0.000000',
  '0.000000',
  '0.000007',
  '0.000007',
  '0.000078',
  '0.000014',
  '0.000000',
  '0.000000',
  '0.000003',
  '0.000000',
  '0.000041',
  '0.000007',
  '0.000000',
  '0.000009',
  '0.000002',
  '0.000000',
  '0.000002',
  '0.000063',
  '0.000020',
  '0.000002',
  '0.000000',
  '0.000000',
  '0.000001',
 

In [12]:
for _,row in data.head(20).iterrows():
    print(row["text"])
    print("actuals: ", row["terms"])
    print("predics: ", infer(model, row["text"]))
    print("="*20)

Mallard. In road, easy to catch, hit by car.  Watched it go under 2 trucks on highway. cracked corn, lettuce, water
actuals:  ['Vehicle collision']
predics:  (['vehicle_collision'], ['0.000009', '0.000003', '0.000001', '0.000001', '0.000000', '0.000000', '0.000011', '0.000003', '0.000001', '0.000000', '0.000001', '0.000000', '0.000002', '0.000002', '0.000000', '0.000000', '0.000000', '0.000000', '0.000000', '0.000003', '0.000001', '0.000001', '0.000001', '0.000000', '0.000007', '0.000003', '0.000000', '0.000005', '0.000000', '0.000000', '0.000000', '0.000009', '0.000001', '0.000000', '0.000005', '0.000002', '0.000000', '0.000000', '0.000011', '0.000004', '0.000000', '0.000001', '0.000032', '0.000000', '0.000004', '0.000000', '0.000032', '0.000000', '0.000000', '0.000001', '0.000000', '0.000003', '0.000000', '0.000000', '0.000001', '0.000003', '0.000000', '0.000004', '0.000001', '0.000000', '0.000004', '0.000001', '0.000000', '0.000001', '0.000005', '0.000003', '0.999970', '0.000000', '